<a href="https://colab.research.google.com/github/Sebi2005/Metaheuristics/blob/main/notebooks/Evolutionary-Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving fi10639.tsp to fi10639.tsp
Saving kroC100.tsp to kroC100.tsp


In [ ]:
import random
import math

def load_data_knapsack(file_name):
    weights_and_values = []
    with open(file_name, 'r') as f:
        lines = f.readlines()
        num_objects = int(lines[0].strip())
        for line in lines[1:-1]:
            parts = line.split()
            if len(parts) >= 3:
                weights_and_values.append((float(parts[1]), float(parts[2])))
        max_capacity = float(lines[-1].strip())
    return weights_and_values, max_capacity

In [ ]:
def is_valid_knapsack(objects, sol, max_capacity):
    total_weight = sum(objects[i][1] for i in range(len(sol)) if sol[i] == 1)
    return total_weight <= max_capacity

def generate_solution_knapsack(n):
    return [random.randint(0, 1) for _ in range(n)]

def generate_valid_solution_knapsack(n, objects, max_capacity):
    while True:
        sol = generate_solution_knapsack(n)
        if is_valid_knapsack(objects, sol, max_capacity):
            return sol

def fitness_knapsack(objects, sol, max_capacity):
    if not is_valid_knapsack(objects, sol, max_capacity):
        return -1
    return sum(objects[i][0] for i in range(len(sol)) if sol[i] == 1)

def generate_rnd_population_knapsack(population_size, problem_size, objects, max_capacity):
    return [generate_valid_solution_knapsack(problem_size, objects, max_capacity) for _ in range(population_size)]

def selection_knapsack(population, population_size, objects, max_capacity):

    indices = random.sample(range(len(population)), 10)

    subset1 = [population[i] for i in indices[:5]]
    subset1.sort(key=lambda sol: fitness_knapsack(objects, sol, max_capacity), reverse=True)
    parent1 = subset1[0]


    subset2 = [population[i] for i in indices[5:]]
    subset2.sort(key=lambda sol: fitness_knapsack(objects, sol, max_capacity), reverse=True)
    parent2 = subset2[0]

    return parent1, parent2


def binary_crossover(population_size, population, crossover_count, objects, max_capacity):
    children = []
    for _ in range(crossover_count):
        p1, p2 = selection_knapsack(population, population_size, objects, max_capacity)
        cut = random.randint(1, len(p1) - 1)
        child1 = p1[:cut] + p2[cut:]
        child2 = p2[:cut] + p1[cut:]
        children.extend([child1, child2])
    return children

def binary_mutation(children, problem_size, mutation_rate):
    mutated_children = []
    for child in children:
        new_child = list(child)
        if random.random() < mutation_rate:
            idx = random.randint(0, problem_size - 1)
            new_child[idx] = 1 - new_child[idx]
        mutated_children.append(new_child)
    return mutated_children


def make_it_valid_rucsac(objects, sol, max_capacity):
    repaired = list(sol)
    for i in range(len(repaired)):
        if is_valid_knapsack(objects, repaired, max_capacity):
            break
        if repaired[i] == 1:
            repaired[i] = 0
    return repaired

def new_population(old_population, mutated_children, objects, max_capacity):
      repaired_children = []
      for child in mutated_children:
          if not is_valid_knapsack(objects, child, max_capacity):
              repaired_children.append(make_it_valid_rucsac(objects, child, max_capacity))
          else:
              repaired_children.append(child)

      combined = old_population + repaired_children
      combined.sort(key=lambda sol: fitness_knapsack(objects, sol, max_capacity), reverse=True)
      return combined[:len(old_population)]

def evaluate_population_knapsack(population_size, population, problem_size, objects, max_capacity):
      fits = [fitness_knapsack(objects, sol, max_capacity) for sol in population]
      fits.sort(reverse=True)
      best = fits[0]
      worst = fits[-1]
      avg = sum(fits) / len(fits)
      return best, avg, worst

def evolutionary_algorithm_knapsack(population_size: int, mutation_rate: float,
                                   crossover_rate: float, generations: int,
                                   problem_size: int, objects: "list[tuple]", max_capacity: int):

      population = generate_rnd_population_knapsack(population_size, problem_size, objects, max_capacity)
      b, a, w = evaluate_population_knapsack(population_size, population, problem_size, objects, max_capacity)
      best = [b]
      avg = [a]
      worst = [w]

      crossover_count = int((population_size * crossover_rate) // 2)

      for i in range(generations):

          children = binary_crossover(population_size, population, crossover_count, objects, max_capacity)

          mutants = binary_mutation(children, problem_size, mutation_rate)

          population = new_population(population, mutants, objects, max_capacity)

          b, a, w = evaluate_population_knapsack(population_size, population, problem_size, objects, max_capacity)
          best.append(b)
          avg.append(a)
          worst.append(w)

      return best, avg, worst



In [ ]:
objects, max_capacity = load_data_knapsack("knapsack-20.txt")
problem_size = len(objects)

pop_size = 50
mut_rate = 0.1
cross_rate = 0.8
gens = 100


print(f"Starting EA for knapsack-20 (Capacity: {max_capacity})...")
best_h, avg_h, worst_h = evolutionary_algorithm_knapsack(
    pop_size, mut_rate, cross_rate, gens, problem_size, objects, max_capacity
)

print(f"Final Results (Generation {gens}):")
print(f"Best Value:    {best_h[-1]}")
print(f"Average Value: {round(avg_h[-1], 2)}")
print(f"Worst Value:   {worst_h[-1]}")

Starting EA for knapsack-20 (Capacity: 524.0)...
Final Results (Generation 100):
Best Value:    716.0
Average Value: 716.0
Worst Value:   716.0


**1. Problem Definition**

The Traveling Salesman Problem (TSP) is a classic combinatorial optimization problem. The goal is to find the shortest possible Hamiltonian cycle—a route that visits a given set of $n$ cities exactly once and returns to the starting city. For large instances like fi10639 (10,639 cities), the search space is $(n-1)!/2$, making metaheuristics like Evolutionary Algorithms essential for finding near-optimal solutions in a reasonable timeframe.

In [ ]:
def read_tsp_file(filename):
    locations = []
    reading_coords = False
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                reading_coords = True
                continue
            if line == "EOF" or not line: break
            if reading_coords:
                parts = line.split()
                locations.append((float(parts[1]), float(parts[2])))
    return locations

In [ ]:
def get_dist(c1, c2):
    return math.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)

def total_route_distance(route, locations):
    d = 0
    for i in range(len(route)):
        d += get_dist(locations[route[i-1]], locations[route[i]])
    return d

**2. Algorithm Used**:

Evolutionary Algorithm (EA)The algorithm mimics the process of natural evolution to iteratively improve a population of candidate routes.

Steps of the Algorithm

Initialisation: A population of $N$ random permutations of city indices is generated.

Evaluation: Each route is evaluated based on its total Euclidean distance (Fitness).

Selection: Parents are chosen using Tournament Selection. Five random individuals are picked, and the one with the shortest distance wins the right to reproduce.

Recombination (Crossover): Two parents produce offspring using specialized operators to ensure valid permutations.

  Ordered Crossover (OX): Preserves the relative order of city sequences. Partially Mapped Crossover

  (PMX): Preserves the absolute position of cities using a mapping system.

Mutation: Offspring undergo Inversion Mutation (reversing a random segment) to maintain genetic diversity and prevent premature convergence.

Replacement: A new population is formed by keeping the best individuals (Elitism) and the new offspring.

In [ ]:
import random
import time
import csv
import math

def ordered_crossover(p1, p2):
    size = len(p1)
    a, b = sorted(random.sample(range(size), 2))
    child = [None] * size
    child[a:b] = p1[a:b]

    p2_pointer = 0
    for i in range(size):
        if child[i] is None:
            while p2[p2_pointer] in child:
                p2_pointer += 1
            child[i] = p2[p2_pointer]
    return child

def pmx_crossover(p1, p2):
    size = len(p1)
    a, b = sorted(random.sample(range(size), 2))
    child = [None] * size
    child[a:b] = p1[a:b]

    for i in range(a, b):
        if p2[i] not in child:
            curr_val = p2[i]
            pos = i
            while a <= pos < b:
                val_at_pos = p1[pos]
                pos = p2.index(val_at_pos)
            child[pos] = curr_val


    for i in range(size):
        if child[i] is None:
            child[i] = p2[i]
    return child

def inversion_mutation(individual, prob):
    if random.random() < prob:
        a, b = sorted(random.sample(range(len(individual)), 2))
        individual[a:b] = individual[a:b][::-1]
    return individual

In [ ]:

def run_tsp_ea(locations, pop_size, generations, mut_rate, cross_type):
    num_cities = len(locations)
    population = [random.sample(range(num_cities), num_cities) for _ in range(pop_size)]

    for gen in range(generations):
        population.sort(key=lambda r: total_route_distance(r, locations))

        new_pop = [population[0]]

        while len(new_pop) < pop_size:
            p1 = min(random.sample(population, 5), key=lambda r: total_route_distance(r, locations))
            p2 = min(random.sample(population, 5), key=lambda r: total_route_distance(r, locations))
            if cross_type == "OX":
                child = ordered_crossover(p1, p2)
            else:
                child = pmx_crossover(p1, p2)


            child = inversion_mutation(child, mut_rate)
            new_pop.append(child)

        population = new_pop

    population.sort(key=lambda r: total_route_distance(r, locations))
    return total_route_distance(population[0], locations)

**3. Parameter Settings**

The following parameters were selected to satisfy the requirement for testing several values while managing the high computational cost of the fi10639 dataset.

| Parameter | Small | Large |
|-----------|-------|-------|
| Pop. size |   10  |  20   |
|Generations|   10  |  20   |

In [ ]:
def compare_tsp_parameters(locations, instances_names):


    configs = [
        {"name": "OX_Small_Pop", "cross": "OX", "pop": 10, "gens": 10, "mut": 0.05},
        {"name": "OX_Large_Pop", "cross": "OX", "pop": 20, "gens": 20, "mut": 0.05},
        {"name": "PMX_Small_Pop", "cross": "PMX", "pop": 10, "gens": 10, "mut": 0.05},
        {"name": "PMX_Large_Pop", "cross": "PMX", "pop": 20, "gens": 20, "mut": 0.05}
    ]

    results = []
    num_runs = 3

    for file_name in instances_names:
        print(f"\n--- Testing Instance: {file_name} ---")

        for cfg in configs:
            print(f"Running Config: {cfg['name']}...")
            run_distances = []
            start_time = time.time()

            for i in range(num_runs):
                dist = run_tsp_ea(
                    locations,
                    pop_size=cfg['pop'],
                    generations=cfg['gens'],
                    mut_rate=cfg['mut'],
                    cross_type=cfg['cross']
                )
                run_distances.append(dist)

            avg_time = (time.time() - start_time) / num_runs
            best_dist = min(run_distances)
            avg_dist = sum(run_distances) / num_runs

            results.append({
                "Instance": file_name,
                "Config_Name": cfg['name'],
                "Crossover": cfg['cross'],
                "Pop_Size": cfg['pop'],
                "Mutation_Rate": cfg['mut'],
                "Best_Distance": round(best_dist, 2),
                "Avg_Distance": round(avg_dist, 2),
                "Avg_Time_Sec": round(avg_time, 4)
            })

    with open('A5_TSP_Results.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    print("\nExperiments complete. Results saved to A5_TSP_Results.csv")
    return results

In [ ]:
def compare_tsp_parameters_kroC(locations, instances_names):


    configs = [
        {"name": "OX_Small_Pop", "cross": "OX", "pop": 10, "gens": 10, "mut": 0.05},
        {"name": "OX_Large_Pop", "cross": "OX", "pop": 20, "gens": 20, "mut": 0.05},
        {"name": "PMX_Small_Pop", "cross": "PMX", "pop": 10, "gens": 10, "mut": 0.05},
        {"name": "PMX_Large_Pop", "cross": "PMX", "pop": 20, "gens": 20, "mut": 0.05}
    ]

    results = []
    num_runs = 3

    for file_name in instances_names:
        print(f"\n--- Testing Instance: {file_name} ---")

        for cfg in configs:
            print(f"Running Config: {cfg['name']}...")
            run_distances = []
            start_time = time.time()

            for i in range(num_runs):
                dist = run_tsp_ea(
                    locations,
                    pop_size=cfg['pop'],
                    generations=cfg['gens'],
                    mut_rate=cfg['mut'],
                    cross_type=cfg['cross']
                )
                run_distances.append(dist)

            avg_time = (time.time() - start_time) / num_runs
            best_dist = min(run_distances)
            avg_dist = sum(run_distances) / num_runs

            results.append({
                "Instance": file_name,
                "Config_Name": cfg['name'],
                "Crossover": cfg['cross'],
                "Pop_Size": cfg['pop'],
                "Mutation_Rate": cfg['mut'],
                "Best_Distance": round(best_dist, 2),
                "Avg_Distance": round(avg_dist, 2),
                "Avg_Time_Sec": round(avg_time, 4)
            })

    with open('A5_TSP_Results_kroC.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    print("\nExperiments complete. Results saved to A5_TSP_Results_kroC.csv")
    return results

In [ ]:
kro_locs = read_tsp_file("kroC100.tsp")
kro_results = compare_tsp_parameters_kroC(kro_locs, ["kroC100.tsp"])


--- Testing Instance: kroC100.tsp ---
Running Config: OX_Small_Pop...
Running Config: OX_Large_Pop...
Running Config: PMX_Small_Pop...
Running Config: PMX_Large_Pop...

Experiments complete. Results saved to A5_TSP_Results_kroC.csv


In [ ]:
kro_locs = read_tsp_file("kroC100.tsp")
fi_locs = read_tsp_file("fi10639.tsp")
kro_results = compare_tsp_parameters(kro_locs, ["kroC100.tsp"])
fi_results = compare_tsp_parameters(fi_locs, ["fi10639.tsp"])


--- Testing Instance: kroC100.tsp ---
Running Config: OX_Small_Pop...
Running Config: OX_Large_Pop...
Running Config: PMX_Small_Pop...
Running Config: PMX_Large_Pop...

Experiments complete. Results saved to A5_TSP_Results.csv

--- Testing Instance: fi10639.tsp ---
Running Config: OX_Small_Pop...
Running Config: OX_Large_Pop...
Running Config: PMX_Small_Pop...
Running Config: PMX_Large_Pop...

Experiments complete. Results saved to A5_TSP_Results.csv


**4. Comparative Results of Experiments**

|Instance|Config\_Name|Crossover|Pop\_Size|Mutation\_Rate|Best\_Distance|Avg\_Distance|Avg\_Time\_Sec|
|---|---|---|---|---|---|---|---|
|kroC100\.tsp|OX\_Small\_Pop|OX|10|0\.05|143928\.87|148598\.98|0\.0609|
|kroC100\.tsp|OX\_Large\_Pop|OX|20|0\.05|119794\.85|128656\.85|0\.2133|
|kroC100\.tsp|PMX\_Small\_Pop|PMX|10|0\.05|142392\.11|149675\.99|0\.0417|
|kroC100\.tsp|PMX\_Large\_Pop|PMX|20|0\.05|128193\.21|132775\.47|0\.1756|

The table below summarizes the performance of the Evolutionary Algorithm across different configurations for the fi10639.tsp instance (10,639 cities). Tests were conducted over 10(small) 20(large) generations.

|Instance|Config\_Name|Crossover|Pop\_Size|Mutation\_Rate|Best\_Distance|Avg\_Distance|Avg\_Time\_Sec|
|---|---|---|---|---|---|---|---|
|fi10639\.tsp|OX\_Small\_Pop|OX|10|0\.05|42094283\.79|42143411\.53|173\.5926|
|fi10639\.tsp|OX\_Large\_Pop|OX|20|0\.05|41656223\.1|41797952\.4|729\.7608|
|fi10639\.tsp|PMX\_Small\_Pop|PMX|10|0\.05|42247090\.52|42261785\.75|41\.9502|
|fi10639\.tsp|PMX\_Large\_Pop|PMX|20|0\.05|41661596\.52|41917984\.61|178\.0094|

**5. Discussion of Results**

**The Efficiency of Ordered Crossover (OX)**

Consistency Across Scales:
In both the 100-city and 10,639-city instances, Ordered Crossover (OX) achieved the absolute lowest Best_Distance when using the larger population size.
Preserving Order: OX is highly effective because it preserves the relative order of cities. In TSP, keeping segments of cities together is more important than their absolute position in the list, which explains why OX consistently beats PMX in final quality.

**The Computational Speed of PMX**

Time Advantage: PMX was consistently faster than OX across all tests. For the large fi10639 instance, PMX was approximately 4x faster than OX (178s vs 729s).

Mapping vs. Searching: PMX utilizes a position-based mapping system that is computationally less expensive than the "searching and reordering" required by OX to fill gaps while avoiding duplicates.

**Conclusion**

For Assignment A5, the experiments clearly determine that Ordered Crossover is the superior operator for route optimization quality. However, PMX is a viable alternative when execution time is a critical constraint. The use of Tournament Selection and Inversion Mutation successfully navigated the vast search spaces of both instances, producing valid Hamiltonian cycles in every run.